In [119]:
import py_vollib.ref_python.black_scholes.greeks.analytical as pyv
import py_vollib.ref_python.black as black
import datetime as dt

def getGreeks(date, expiry, stockPrice, r, sigma, strike):
    flag = 'p'
    dateDt = dt.datetime.strptime(date, "%Y-%m-%d")
    expiryDt = dt.datetime.strptime(expiry, "%Y-%m-%d")

    t = (expiryDt-dateDt).days/365.25

    delta = pyv.delta(flag, stockPrice, strike, t, r, sigma)
    gamma = pyv.gamma(flag, stockPrice, strike, t, r, sigma)
    theta = pyv.theta(flag, stockPrice, strike, t, r, sigma)
    vega = pyv.vega(flag, stockPrice, strike, t, r, sigma)
    rho  = pyv.rho(flag, stockPrice, strike, t, r, sigma)
    return delta, gamma, vega, theta, rho

def get_delta(date, expiry, stockPrice, r, sigma, strike):
    flag = 'p'
    dateDt = dt.datetime.strptime(date, "%Y-%m-%d")
    expiryDt = dt.datetime.strptime(expiry, "%Y-%m-%d")

    t = (expiryDt-dateDt).days/365.25
    return pyv.delta(flag, stockPrice, strike, t, r, sigma)

In [120]:
import yfinance as yf
from dateutil.relativedelta import relativedelta

start = dt.datetime(year=2021, month=1, day=1)

yf.Ticker("AAPL").history(start=start, interval="1d").index[0]

Timestamp('2021-01-04 00:00:00-0500', tz='America/New_York')

In [121]:
def quarter_start(start):
    return yf.Ticker("AAPL").history(start=start, interval="1d").index[0]

def quarter_end(start):
    end = start+relativedelta(months=3)
    return yf.Ticker("AAPL").history(end=end, interval="1d").index[-1]

In [127]:
start_date, end_date = quarter_start(start).strftime('%Y-%m-%d'), quarter_end(start).strftime('%Y-%m-%d')
strike = 180
ticker = 'BA'

In [128]:
from utils.data import *

start = dt.datetime(year=2000, month=4, day=1)
end = dt.datetime(year=2026, month=10, day=1)
all_stocks = ["NVDA", "AAPL", "BA", 'DB']

n = len(all_stocks)

pce = load_pce()
effr = load_ffr()

a = (0.005, 4)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

date = dt.datetime(year=2026, month=4, day=1)
exp = dt.datetime(year=2026, month=7, day=1)

price = yf.Ticker(all_stocks[0]).history(start=date, period='1d').Open.iloc[0]

qs = quarter_start(start).strftime("%Y-%m-%d")
qe = quarter_end(start).strftime("%Y-%m-%d")

def get_strike_from_delta(target_delta, initial_guess, qs, qe, r, stock_price, sigma):
    x1, x2 = (0, initial_guess)
    finished = False
    
    tol = 1e-3
    while not finished:
        midpoint = (x1+x2)/2
        
        delta = -get_delta(qs, qe, stock_price, r, sigma, midpoint)
        
        if target_delta-delta > 0:
            x1 = midpoint
        else:
            x2 = midpoint
    
        if abs(target_delta - delta) < tol: finished= True
    return x1/2+x2/2


In [129]:
strike = get_strike_from_delta(0.5, price*2, qs, qe, 0.05, price, Sigma[0,0]*2)

#put_price = black.black_put(price, strike, 1, 0.05, Sigma[0,0]*2)

In [139]:
import plotly.express as px

#def history_with_put(ticker, start, end, strike):
df = pd.DataFrame(columns =['Open', 'price_with_put'])
ticker = "WMT"
for q in range(16):
    period_start = (dt.datetime(year=2021, month=1, day=1) + relativedelta(months=3*q)).strftime("%Y-%m-%d")
    period_end = (dt.datetime(year=2021, month=4, day=1) + relativedelta(months=3*q)).strftime("%Y-%m-%d")

    hist = yf.Ticker(ticker).history(start=period_start, end=period_end, interval="1d")[['Open']]

    
    hist["price_with_put"] = hist.Open - hist.Open.iloc[0] + previous_value if q else hist.Open


    strike = get_strike_from_delta(0.5, price*2, period_start, period_end, 0.05, hist.Open.iloc[0], Sigma[0,0]*2)
    
    num_days = len(hist.index)

    put_value = np.array([black.black_put(hist.Open.iloc[i], strike, (num_days-i)/365.25, 0.05, Sigma[0,0]*2) for i in range(num_days)])

    put_value -= put_value[0]

    hist.price_with_put += put_value

    previous_value = hist.price_with_put.iloc[-1]
    df = hist if not len(df.index) else pd.concat([df, hist])
   # px.line(hist).show()

In [140]:
px.line(df)

In [86]:
hist

,Open,price_with_put
Date,,
2024-01-02 00:00:00-05:00,185.225777,185.225777
2024-01-03 00:00:00-05:00,182.325885,183.267950
2024-01-04 00:00:00-05:00,180.277180,181.980868
2024-01-05 00:00:00-05:00,180.118838,181.836461
2024-01-08 00:00:00-05:00,180.217821,181.840032
...,...,...
2024-03-22 00:00:00-04:00,170.210928,174.428079
2024-03-25 00:00:00-04:00,169.031685,174.363936
2024-03-26 00:00:00-04:00,168.466821,174.348180
